In [1]:
# 1. Force remove the wrong 'edgar' package and alternatives
%pip uninstall edgar python-edgar sec-api -y

# 2. Install the correct package (it must be 'edgartools')
%pip install edgartools

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
# %pip uninstall edgar -y
# ! pip install edgar

# ! pip install sec-api

from edgar import *
set_identity("Jason Rodriguez jabob2002@gmail.com")
import pandas as pd
import numpy as np

c:\Users\jabob\.conda\conda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\jabob\.conda\conda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
#  test 


amex = Company("AXP")


cik = amex.cik

files = amex.get_filings(form="10-Q").head(10)

print(cik)

4962


In [4]:
apple = Company("AAPL")


apple_cik = amex.cik

files = apple.get_filings(form="10-Q").head(10)

print(cik)

4962


In [22]:
import pandas as pd

df = files.to_pandas()

df.columns = df.columns.str.lower()

time_stamps_columns = ['filing_date', 'reportdate']


quarter_col = 'reportdate'

df[quarter_col] = pd.to_datetime(df[quarter_col])
df['quarter_year'] = (
    df['reportdate'].dt.year.astype(str) 
    + "'Q" 
    + df['reportdate'].dt.quarter.astype(str)
)

df

,accession_number,filing_date,reportdate,acceptancedatetime,act,form,filenumber,items,size,isxbrl,isinlinexbrl,primarydocument,primarydocdescription,quarter_year
0,0000320193-26-000013,2026-05-01,2026-03-28,2026-05-01 10:01:00+00:00,34,10-Q,001-36743,,5809851,1,1,aapl-20260328.htm,10-Q,2026'Q1
1,0000320193-26-000006,2026-01-30,2025-12-27,2026-01-30 11:01:32+00:00,34,10-Q,001-36743,,5191740,1,1,aapl-20251227.htm,10-Q,2025'Q4
2,0000320193-25-000073,2025-08-01,2025-06-28,2025-08-01 10:00:42+00:00,34,10-Q,001-36743,,5304776,1,1,aapl-20250628.htm,10-Q,2025'Q2
3,0000320193-25-000057,2025-05-02,2025-03-29,2025-05-02 10:00:46+00:00,34,10-Q,001-36743,,5299807,1,1,aapl-20250329.htm,10-Q,2025'Q1
4,0000320193-25-000008,2025-01-31,2024-12-28,2025-01-31 11:01:27+00:00,34,10-Q,001-36743,,5150277,1,1,aapl-20241228.htm,10-Q,2024'Q4
5,0000320193-24-000081,2024-08-02,2024-06-29,2024-08-01 22:03:34+00:00,34,10-Q,001-36743,,5372771,1,1,aapl-20240629.htm,10-Q,2024'Q2
6,0000320193-24-000069,2024-05-03,2024-03-30,2024-05-02 22:04:25+00:00,34,10-Q,001-36743,,5284139,1,1,aapl-20240330.htm,10-Q,2024'Q1
7,0000320193-24-000006,2024-02-02,2023-12-30,2024-02-01 23:03:38+00:00,34,10-Q,001-36743,,4984121,1,1,aapl-20231230.htm,10-Q,2023'Q4
8,0000320193-23-000077,2023-08-04,2023-07-01,2023-08-03 22:04:43+00:00,34,10-Q,001-36743,,5939898,1,1,aapl-20230701.htm,10-Q,2023'Q3
9,0000320193-23-000064,2023-05-05,2023-04-01,2023-05-04 22:03:52+00:00,34,10-Q,001-36743,,6314786,1,1,aapl-20230401.htm,10-Q,2023'Q2


In [96]:
def fetching_reportings(ticker):
    all_dfs = []

    ticker_var = Company(ticker)
    ticker_cik = ticker_var.cik
    files = ticker_var.get_filings(form=["10-Q", "10-K"])

    for filing in files:
        try:
            xbrl = filing.xbrl()
            if not xbrl:
                continue
                
            income_statement = xbrl.statements.income_statement()
            if income_statement is None:
                continue

            df = income_statement.to_dataframe()
            df = df[df['abstract'] == False].copy()
            
            date_cols = [col for col in df.columns if col[0].isdigit()]
            df = df[['label'] + date_cols]
            df['label'] = df['label'].str.strip().str.replace(':', '')

            df = df.drop_duplicates(subset='label', keep='first')
            df = df.set_index('label')

            df_T = df.T
            df_T.index.name = 'period'
            df_T = df_T.reset_index()
            
            df_T['date'] = pd.to_datetime(df_T['period'].str.extract(r'(\d{4}-\d{2}-\d{2})')[0])
            df_T['period_type'] = df_T['period'].str.extract(r'\((\w+)\)')[0]

            # ---- CHANGE IS HERE ----
            df_T['period_type'] = df_T['period_type'].fillna(
                'Q4' if filing.form == '10-K' else 'SKIP'
            )
            # ------------------------

            df_T = df_T[df_T['period_type'].isin(['Q1', 'Q2', 'Q3', 'Q4'])]
            df_T = df_T.drop(columns=['period', 'period_type']).set_index('date')
            df_T = df_T.apply(pd.to_numeric, errors='coerce')
            
            all_dfs.append(df_T)

        except Exception as e:
            print(f"Failed processing {filing.company} ({filing.filing_date}): {e}")

    df_final = pd.concat(all_dfs)
    df_final = df_final.loc[:, ~df_final.columns.duplicated(keep='first')]
    df_final = df_final[~df_final.index.duplicated(keep='first')]
    df_final = df_final.sort_index()

    df_final.columns = df_final.columns.str.lower().str.replace(' ', '_')
    return df_final

data = fetching_reportings('AAPL')

print(data)


Filing(company='Apple Inc.', cik=320193, form='10-K/A', filing_date='2010-01-25', accession_no='0001193125-10-012091')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`


Filing(company='Apple Inc.', cik=320193, form='10-Q/A', filing_date='2009-04-27', accession_no='0001193125-09-087629')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`

No XBRL attachments found in filing Filing(company='Apple Inc.', cik=320193, form='10-Q/A', filing_date='2009-04-27', accession_no='0001193125-09-087629')
No XBRL attachments found in filing Filing(company='Apple Inc.', cik=320193, form='10-Q', filing_date='2009-04-23', accession_no='0001193125-09-085781')
No XBRL attachments found in filing Filing(compa

label          net_sales      products        iphone           mac  \
date                                                                 
2008-06-28           NaN           NaN           NaN           NaN   
2008-12-27           NaN           NaN           NaN           NaN   
2009-03-28           NaN           NaN           NaN           NaN   
2009-06-27  9.734000e+09           NaN           NaN           NaN   
2009-12-26  1.568300e+10           NaN           NaN           NaN   
2010-03-27  1.349900e+10           NaN           NaN           NaN   
2010-06-26  1.570000e+10           NaN           NaN           NaN   
2010-12-25  2.674100e+10           NaN           NaN           NaN   
2011-03-26  2.466700e+10           NaN           NaN           NaN   
2011-06-25  2.857100e+10           NaN           NaN           NaN   
2011-12-31  4.633300e+10           NaN           NaN           NaN   
2012-03-31  3.918600e+10           NaN           NaN           NaN   
2012-06-30  3.502300

In [101]:
def calculating_margins(data): 
    df_margins = None 
    
    try:
        margins = ['net_sales', 'gross_margin', 'net_income']
        
        if set(margins).issubset(data.columns):
            
            df_margins = data[margins].copy()

            df_margins['gross_margin_rate'] = (df_margins['gross_margin'] / df_margins['net_sales']) * 100
            df_margins['net_margin_rate'] = (df_margins['net_income'] / df_margins['net_sales']) * 100

            df_margins.dropna(axis=0, inplace=True)
            
            df_margins['bps_difference'] = (df_margins['gross_margin_rate'] - df_margins['net_margin_rate']) 
            df_margins['dollar_difference'] = df_margins['gross_margin'] - df_margins['net_income'] 
        else:
            print("yikes. One or more required columns are missing from the DataFrame.") 
            
    # Catching generic Exceptions ensures you see math/key errors, not just NameErrors
    except Exception as e:
        print(f"Error during calculation: {e}")

    return df_margins

apple_margins = calculating_margins(data)

apple_margins

label,net_sales,gross_margin,net_income,gross_margin_rate,net_margin_rate,bps_difference,dollar_difference
date,,,,,,,
2009-06-27,9.734000e+09,3.983000e+09,1.828000e+09,40.918430,18.779536,22.138895,2.155000e+09
2009-12-26,1.568300e+10,6.411000e+09,3.378000e+09,40.878658,21.539246,19.339412,3.033000e+09
2010-03-27,1.349900e+10,5.625000e+09,3.074000e+09,41.669753,22.772057,18.897696,2.551000e+09
2010-06-26,1.570000e+10,6.136000e+09,3.253000e+09,39.082803,20.719745,18.363057,2.883000e+09
2010-12-25,2.674100e+10,1.029800e+10,6.004000e+09,38.510153,22.452414,16.057739,4.294000e+09
2011-03-26,2.466700e+10,1.021800e+10,5.987000e+09,41.423765,24.271294,17.152471,4.231000e+09
2011-06-25,2.857100e+10,1.192200e+10,7.308000e+09,41.727626,25.578384,16.149242,4.614000e+09
2011-12-31,4.633300e+10,2.070300e+10,1.306400e+10,44.683055,28.195886,16.487169,7.639000e+09
2012-03-31,3.918600e+10,1.856400e+10,1.162200e+10,47.374062,29.658552,17.715511,6.942000e+09


In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(20, 8))

# df_margins.plot(y='gross_margin', ax=plt.gca(), linewidth=2)
# plt.title('Gross Margin Evolution Across Financial Quarters', fontsize=16, fontweight='bold', pad=15)
# plt.xlabel('Report Date / Quarter', fontsize=12)
# plt.ylabel('Gross Margin Value', fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.6) # Adds a light background grid
# plt.tight_layout()
# plt.show()

# df_margins.plot(y ='net_margin')

label,net_sales,gross_margin,net_income,gross_margin_rate,net_margin_rate,bps_difference,dollar_difference
date,,,,,,,
2009-06-27,9.734000e+09,3.983000e+09,1.828000e+09,40.918430,18.779536,22.138895,2.155000e+09
2009-12-26,1.568300e+10,6.411000e+09,3.378000e+09,40.878658,21.539246,19.339412,3.033000e+09
2010-03-27,1.349900e+10,5.625000e+09,3.074000e+09,41.669753,22.772057,18.897696,2.551000e+09
2010-06-26,1.570000e+10,6.136000e+09,3.253000e+09,39.082803,20.719745,18.363057,2.883000e+09
2010-12-25,2.674100e+10,1.029800e+10,6.004000e+09,38.510153,22.452414,16.057739,4.294000e+09
2011-03-26,2.466700e+10,1.021800e+10,5.987000e+09,41.423765,24.271294,17.152471,4.231000e+09
2011-06-25,2.857100e+10,1.192200e+10,7.308000e+09,41.727626,25.578384,16.149242,4.614000e+09
2011-12-31,4.633300e+10,2.070300e+10,1.306400e+10,44.683055,28.195886,16.487169,7.639000e+09
2012-03-31,3.918600e+10,1.856400e+10,1.162200e+10,47.374062,29.658552,17.715511,6.942000e+09
